# 2 顺序调度 SequentialOrchestration

In [55]:
import builtins
from rich import print
from dotenv import load_dotenv
from IPython.display import Markdown, display
import asyncio

In [2]:
load_dotenv("../.env/qiniu", override=True)

True

In [ ]:
from semantic_kernel.agents import ChatCompletionAgent, SequentialOrchestration
from semantic_kernel.contents import ChatMessageContent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.agents.runtime import InProcessRuntime

In [ ]:
concept_extractor_agent = ChatCompletionAgent(
    name="ConceptExtractorAgent",
    instructions=(
        "You are a marketing analyst. Given a product description, identify:\n"
        "- Key features\n"
        "- Target audience\n"
        "- Unique selling points\n\n"
    ),
    service=OpenAIChatCompletion(),
)
writer_agent = ChatCompletionAgent(
    name="WriterAgent",
    instructions=(
        "You are a marketing copywriter. Given a block of text describing features, audience, and USPs, "
        "compose a compelling marketing copy (like a newsletter section) that highlights these points. "
        "Output should be short (around 150 words), output just the copy as a single text block."
    ),
    service=OpenAIChatCompletion(),
)
format_proof_agent = ChatCompletionAgent(
    name="FormatProofAgent",
    instructions=(
        "You are an editor. Given the draft copy, correct grammar, improve clarity, ensure consistent tone, "
        "give format and make it polished. Output the final improved copy as a single text block."
    ),
    service=OpenAIChatCompletion(),
)

In [50]:
agents = [concept_extractor_agent, writer_agent, format_proof_agent]

In [13]:
def agent_response_callback(message: ChatMessageContent) -> None:
    """Observer function to print the messages from the agents."""
    name = message.name.lower()
    if "proof" in name:
        print(f"[#E74C3C]{message.name} ▶︎ {message.content}[/#E74C3C]")
    elif "writer" in name:
        print(f"[#2ECC71]{message.name} ▶︎ {message.content}[/#2ECC71]")
    elif "extractor" in name:
        print(f"[#F1C40F]{message.name} ▶︎ {message.content}[/#F1C40F]")
    else:
        print(f"[#9B59B6]{message.name} ▶︎ {message.content}[/#9B59B6]")

In [ ]:
sor = SequentialOrchestration(
    members=agents,
    agent_response_callback=agent_response_callback,
)

In [15]:
# 2. Create a runtime and start it
runtime = InProcessRuntime()
runtime.start()

orchestration_result = await sor.invoke(
    task="An eco-friendly stainless steel water bottle that keeps drinks cold for 24 hours",
    runtime=runtime,
)
value = await orchestration_result.get(timeout=200)


ConceptExtractorAgent ▶︎ ## Product Analysis

### Key Features
- **Eco-friendly design** – Reusable, sustainable alternative to single-use plastics
- **Stainless steel construction** – Durable, long-lasting material
- **24-hour cold retention** – Advanced insulation keeps beverages cold all day

### Target Audience
- **Environmentally conscious consumers** – People actively trying to reduce their plastic footprint
- **Fitness enthusiasts & athletes** – Those who need hydration during workouts and throughout the day
- **Outdoor adventurers** – Hikers, campers, travelers who need reliable beverage temperature
- **Everyday commuters** – Professionals wanting cold water at the office or on the go
- **Health-conscious individuals** – People who prioritize staying hydrated

### Unique Selling Points
- **Extended temperature control** – 24-hour cold retention outperforms many competitors
- **Sustainability + performance** – Rare combination of eco-friendliness with high functional performance
- **Durability** – Stainless steel offers longevity that justifies the investment over disposable bottles
- **Zero plastic** – Appeals to consumers wanting to eliminate single-use plastics entirely

---

Would you like me to suggest marketing messaging or positioning strategies based on this analysis?

WriterAgent ▶︎ Stay Hydrated. Stay Sustainable. One Bottle Does It All.

Meet your new hydration essential—designed for the planet, built for your life.

Whether you're crushing it at the gym, conquering a trail, or powering through your workday, you need a bottle that
keeps up. Ours delivers 24-hour cold retention—outperforming the competition—so your water stays refreshing from 
morning meetings to evening adventures.

Crafted from premium stainless steel, it's built to last a lifetime—no more replacing flimsy plastic bottles. And 
here's the best part: this is a zero-plastic, reusable solution that helps you shrink your environmental footprint 
without sacrificing performance.

Because why choose between sustainability and functionality when you can have both?

Upgrade your hydration habit. Make the switch today.

FormatProofAgent ▶︎ **Stay Hydrated. Stay Sustainable. One Bottle Does It All.**

Meet your new hydration essential—designed for the planet, built for your life.

Whether you're powering through a workout, tackling a trail, or conquering your workday, you need a bottle that 
keeps up. Ours delivers 24-hour cold retention—outperforming the competition—so your water stays refreshing from 
morning meetings to evening adventures.

Crafted from premium stainless steel, it's built to last a lifetime—no more replacing flimsy plastic bottles. This 
is a zero-plastic, reusable solution that helps you shrink your environmental footprint without sacrificing 
performance.

Why choose between sustainability and functionality when you can have both?

**Upgrade your hydration habit. Make the switch today.**

In [24]:
print("##### Final Result #####")
display(Markdown(value.content))

##### Final Result #####

**Stay Hydrated. Stay Sustainable. One Bottle Does It All.**

Meet your new hydration essential—designed for the planet, built for your life.

Whether you're powering through a workout, tackling a trail, or conquering your workday, you need a bottle that keeps up. Ours delivers 24-hour cold retention—outperforming the competition—so your water stays refreshing from morning meetings to evening adventures.

Crafted from premium stainless steel, it's built to last a lifetime—no more replacing flimsy plastic bottles. This is a zero-plastic, reusable solution that helps you shrink your environmental footprint without sacrificing performance.

Why choose between sustainability and functionality when you can have both?

**Upgrade your hydration habit. Make the switch today.**

In [25]:
await runtime.stop_when_idle()

# 2a: 2 + .cancel()

In [ ]:
sor2 = SequentialOrchestration(members=agents)

In [44]:
runtime = InProcessRuntime()
runtime.start()

In [45]:
sor2_result = await sor2.invoke(
    task="An eco-friendly stainless steel water bottle that keeps drinks cold for 24 hours",
    runtime=runtime,
)

In [47]:
await asyncio.sleep(1)
sor2_result.cancel()
try:
    value = await orchestration_result.get(timeout=20)
    print(value)
except Exception as e:
    print(e)

RuntimeError: The invocation has already been canceled.

In [48]:
await runtime.stop_when_idle()

# 2b: 2 + streaming

In [51]:
from semantic_kernel.contents import StreamingChatMessageContent

In [57]:
is_new_message = True


def streaming_agent_response_callback(
    message: StreamingChatMessageContent, is_final: bool
) -> None:
    """Observer function to print the messages from the agents.

    Args:
        message (StreamingChatMessageContent): The streaming message content from the agent.
        is_final (bool): Indicates if this is the final part of the message.
    """
    global is_new_message
    if is_new_message:
        builtins.print(f"# {message.name}")
        is_new_message = False
    builtins.print(message.content, end="", flush=True)
    if is_final:
        builtins.print()
        is_new_message = True

In [58]:
sor3 = SequentialOrchestration(
    members=agents,
    streaming_agent_response_callback=streaming_agent_response_callback,
)

In [59]:
runtime = InProcessRuntime()
runtime.start()

orchestration_result = await sor3.invoke(
    task="An eco-friendly stainless steel water bottle that keeps drinks cold for 24 hours",
    runtime=runtime,
)

value = await orchestration_result.get(timeout=200)

# ConceptExtractorAgent
## Product Analysis

**Product:** Eco-friendly stainless steel water bottle with 24-hour cold retention

---

### Key Features

- **Material:** Stainless steel construction (durable, long-lasting)
- **Insulation:** Keeps drinks cold for 24 hours
- **Eco-friendly:** Reusable, reduces single-use plastic waste
- **Sustainability:** Designed for repeated use

---

### Target Audience

- **Eco-conscious consumers** – People actively trying to reduce their plastic footprint
- **Outdoor enthusiasts** – Hikers, campers, travelers who need reliable hydration
- **Fitness enthusiasts** – Gym-goers, runners, athletes
- **Everyday commuters** – Professionals seeking convenient, sustainable hydration
- **Gen Z & Millennials** – Demographics with strong sustainability values

---

### Unique Selling Points (USPs)

1. **24-hour cold retention** – Superior insulation outperforms most competitors
2. **Eco-friendly positioning** – Appeals to growing sustainability movement
3. **Du

In [60]:
print("***** Final Result *****")
print(f"[#9B59B6]{value}[/#9B59B6]")

***** Final Result *****

Stay Cold. Stay Sustainable.

Meet your new hydration essential—the water bottle that keeps drinks ice-cold for 24 hours, no matter where life 
takes you. Whether you're crushing a morning workout, conquering a trail, or powering through your commute, this 
bottle is built to perform.

Crafted from premium stainless steel, it's virtually indestructible—meaning one bottle replaces thousands of 
disposable plastics. No more worrying about chemicals leaching into your water. No more last-minute convenience 
store runs. Just pure, refreshing hydration whenever you need it.

Here's the thing: you already care about the planet. This bottle makes it effortless. Every sip is a small victory 
against single-use waste. Every refill saves you money while doing right by the environment.

Whether you're a fitness fanatic, an outdoor adventurer, or just someone who wants to make better choices, this 
bottle was made for you.

**Cooler drinks. Cleaner conscience. One purchase. Endless impact.**

In [61]:
await runtime.stop_when_idle()